# S3 - Procesamiento y Calidad de Datos
## Proyecto Sello — Contaminación del Agua

Actividad individual (sección 4 de la guía S3), aplicada a `lecturas_agua.csv`
(1,500,000 lecturas de sensores de agua en 4 puntos: Río Coata, Juliaca (urbano),
Cabanillas (ciudad) y Cabanillas (nacimiento de agua)).

Controles de calidad aplicados en esta sesión: validación de esquema, detección y
tratamiento de nulos, identificación y tratamiento de duplicados, y escritura de una
salida analítica particionada en Parquet (verificada por lectura y por `explain()`).


## 1. SparkSession

In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("proyecto-agua-calidad-datos")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark


In [6]:
ORIGEN_DATOS = "/opt/s03-procesamiento-calidad-datos/data"
ARTIFACTS = "/opt/s03-procesamiento-calidad-datos/artifacts"


## 2. Cargar con esquema explícito y validar columnas [Bronze → control de calidad #1]

Esquema explícito (igual que en S2, `sensor_id` se mantiene como texto para no perder
el cero inicial). Nuevo en S3: además de tipar bien, se verifica que las columnas
obligatorias existan de verdad — si falta alguna, el pipeline se detiene ahí mismo.

In [7]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

schema_agua = StructType([
    StructField("id_lectura", IntegerType(), True),
    StructField("sensor_id", StringType(), True),
    StructField("ubicacion", StringType(), True),
    StructField("fecha_hora", TimestampType(), True),
    StructField("canal_transmision_id", IntegerType(), True),
    StructField("ph", DoubleType(), True),
    StructField("turbidez_ntu", DoubleType(), True),
    StructField("temperatura_c", DoubleType(), True),
    StructField("conductividad_us_cm", DoubleType(), True),
    StructField("solidos_disueltos_totales_mg_l", DoubleType(), True),
    StructField("oxigeno_disuelto_mg_l", DoubleType(), True),
    StructField("plomo_mg_l", DoubleType(), True),
    StructField("arsenico_mg_l", DoubleType(), True),
    StructField("mercurio_mg_l", DoubleType(), True),
    StructField("cadmio_mg_l", DoubleType(), True),
    StructField("coliformes_fecales_nmp_100ml", DoubleType(), True),
    StructField("escherichia_coli_nmp_100ml", DoubleType(), True),
    StructField("presencia_parasitos", IntegerType(), True),
    StructField("radiactividad_bq_l", DoubleType(), True),
    StructField("caudal_l_s", DoubleType(), True),
    StructField("indice_riesgo_normalizado", DoubleType(), True),
])

df_agua = spark.read.csv(
    f"{ORIGEN_DATOS}/lecturas_agua.csv",
    header=True,
    schema=schema_agua,
)

df_agua.printSchema()


root
 |-- id_lectura: integer (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- ubicacion: string (nullable = true)
 |-- fecha_hora: timestamp (nullable = true)
 |-- canal_transmision_id: integer (nullable = true)
 |-- ph: double (nullable = true)
 |-- turbidez_ntu: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- conductividad_us_cm: double (nullable = true)
 |-- solidos_disueltos_totales_mg_l: double (nullable = true)
 |-- oxigeno_disuelto_mg_l: double (nullable = true)
 |-- plomo_mg_l: double (nullable = true)
 |-- arsenico_mg_l: double (nullable = true)
 |-- mercurio_mg_l: double (nullable = true)
 |-- cadmio_mg_l: double (nullable = true)
 |-- coliformes_fecales_nmp_100ml: double (nullable = true)
 |-- escherichia_coli_nmp_100ml: double (nullable = true)
 |-- presencia_parasitos: integer (nullable = true)
 |-- radiactividad_bq_l: double (nullable = true)
 |-- caudal_l_s: double (nullable = true)
 |-- indice_riesgo_normalizado: double (nullab

In [8]:
columnas_requeridas = {
    "id_lectura", "sensor_id", "ubicacion", "fecha_hora",
    "ph", "plomo_mg_l", "arsenico_mg_l", "mercurio_mg_l",
    "presencia_parasitos", "radiactividad_bq_l",
}

faltantes = columnas_requeridas - set(df_agua.columns)
if faltantes:
    raise ValueError(f"Faltan columnas obligatorias: {sorted(faltantes)}")

print("Esquema validado: las columnas requeridas estan presentes.")
print(df_agua.columns)
df_agua.count()


Esquema validado: las columnas requeridas estan presentes.
['id_lectura', 'sensor_id', 'ubicacion', 'fecha_hora', 'canal_transmision_id', 'ph', 'turbidez_ntu', 'temperatura_c', 'conductividad_us_cm', 'solidos_disueltos_totales_mg_l', 'oxigeno_disuelto_mg_l', 'plomo_mg_l', 'arsenico_mg_l', 'mercurio_mg_l', 'cadmio_mg_l', 'coliformes_fecales_nmp_100ml', 'escherichia_coli_nmp_100ml', 'presencia_parasitos', 'radiactividad_bq_l', 'caudal_l_s', 'indice_riesgo_normalizado']


1500000

## 3. Explorar nulos por columna [control de calidad #2]

In [9]:
from pyspark.sql.functions import col, count, when

total_filas = df_agua.count()

nulos = df_agua.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_agua.columns
]).collect()[0].asDict()

for columna, cantidad in nulos.items():
    porcentaje = cantidad / total_filas * 100
    print(f"{columna}: {cantidad} nulos ({porcentaje:.1f}%)")


[Stage 6:===>                                                     (1 + 15) / 16]

id_lectura: 0 nulos (0.0%)
sensor_id: 0 nulos (0.0%)
ubicacion: 0 nulos (0.0%)
fecha_hora: 0 nulos (0.0%)
canal_transmision_id: 0 nulos (0.0%)
ph: 0 nulos (0.0%)
turbidez_ntu: 0 nulos (0.0%)
temperatura_c: 0 nulos (0.0%)
conductividad_us_cm: 0 nulos (0.0%)
solidos_disueltos_totales_mg_l: 0 nulos (0.0%)
oxigeno_disuelto_mg_l: 60000 nulos (4.0%)
plomo_mg_l: 0 nulos (0.0%)
arsenico_mg_l: 0 nulos (0.0%)
mercurio_mg_l: 0 nulos (0.0%)
cadmio_mg_l: 0 nulos (0.0%)
coliformes_fecales_nmp_100ml: 60000 nulos (4.0%)
escherichia_coli_nmp_100ml: 0 nulos (0.0%)
presencia_parasitos: 0 nulos (0.0%)
radiactividad_bq_l: 0 nulos (0.0%)
caudal_l_s: 0 nulos (0.0%)
indice_riesgo_normalizado: 0 nulos (0.0%)


`isNull()` no detecta cadenas vacías. Se confirma también sobre la columna crítica `sensor_id`:

In [10]:
from pyspark.sql.functions import trim

df_agua.filter(
    col("sensor_id").isNull() | (trim(col("sensor_id")) == "")
).count()


0

## 4. Filtrado de datos (`filter()`/`where()`)

In [11]:
# Expresión SQL como texto
df_agua.filter("ubicacion = 'Rio Coata'").show(5, truncate=False)
df_agua.filter("ph BETWEEN 6.5 AND 8.5").count()


+----------+---------+---------+-------------------+--------------------+----+------------+-------------+-------------------+------------------------------+---------------------+----------+-------------+-------------+-----------+----------------------------+--------------------------+-------------------+------------------+----------+-------------------------+
|id_lectura|sensor_id|ubicacion|fecha_hora         |canal_transmision_id|ph  |turbidez_ntu|temperatura_c|conductividad_us_cm|solidos_disueltos_totales_mg_l|oxigeno_disuelto_mg_l|plomo_mg_l|arsenico_mg_l|mercurio_mg_l|cadmio_mg_l|coliformes_fecales_nmp_100ml|escherichia_coli_nmp_100ml|presencia_parasitos|radiactividad_bq_l|caudal_l_s|indice_riesgo_normalizado|
+----------+---------+---------+-------------------+--------------------+----+------------+-------------+-------------------+------------------------------+---------------------+----------+-------------+-------------+-----------+----------------------------+------------------

1065553

In [12]:
# Expresión booleana con col()
df_agua.filter(col("ubicacion") == "Rio Coata").show(5, truncate=False)
df_agua.filter(col("ph").between(6.5, 8.5)).count()


+----------+---------+---------+-------------------+--------------------+----+------------+-------------+-------------------+------------------------------+---------------------+----------+-------------+-------------+-----------+----------------------------+--------------------------+-------------------+------------------+----------+-------------------------+
|id_lectura|sensor_id|ubicacion|fecha_hora         |canal_transmision_id|ph  |turbidez_ntu|temperatura_c|conductividad_us_cm|solidos_disueltos_totales_mg_l|oxigeno_disuelto_mg_l|plomo_mg_l|arsenico_mg_l|mercurio_mg_l|cadmio_mg_l|coliformes_fecales_nmp_100ml|escherichia_coli_nmp_100ml|presencia_parasitos|radiactividad_bq_l|caudal_l_s|indice_riesgo_normalizado|
+----------+---------+---------+-------------------+--------------------+----+------------+-------------+-------------------+------------------------------+---------------------+----------+-------------+-------------+-----------+----------------------------+------------------

1065553

`.between()` es inclusivo; una comparación estricta escrita a mano no da el mismo resultado:

In [13]:
print("between (incluye 6.5 y 8.5):", df_agua.filter(col("ph").between(6.5, 8.5)).count())
print("estricto (excluye 6.5 y 8.5):   ", df_agua.filter((col("ph") > 6.5) & (col("ph") < 8.5)).count())


between (incluye 6.5 y 8.5): 1065553


[Stage 23:===>                                                    (1 + 15) / 16]

estricto (excluye 6.5 y 8.5):    1057946


In [14]:
# eqNullSafe(): trata null como valor comparable
df_agua.filter(col("presencia_parasitos").eqNullSafe(None)).count()


0

In [15]:
# where() es alias de filter()
df_agua.where(col("radiactividad_bq_l") > 0.5).show(5, truncate=False)


+----------+---------+---------+-------------------+--------------------+----+------------+-------------+-------------------+------------------------------+---------------------+----------+-------------+-------------+-----------+----------------------------+--------------------------+-------------------+------------------+----------+-------------------------+
|id_lectura|sensor_id|ubicacion|fecha_hora         |canal_transmision_id|ph  |turbidez_ntu|temperatura_c|conductividad_us_cm|solidos_disueltos_totales_mg_l|oxigeno_disuelto_mg_l|plomo_mg_l|arsenico_mg_l|mercurio_mg_l|cadmio_mg_l|coliformes_fecales_nmp_100ml|escherichia_coli_nmp_100ml|presencia_parasitos|radiactividad_bq_l|caudal_l_s|indice_riesgo_normalizado|
+----------+---------+---------+-------------------+--------------------+----+------------+-------------+-------------------+------------------------------+---------------------+----------+-------------+-------------+-----------+----------------------------+------------------

## 5. Ordenar resultados (`orderBy()`/`sort()`)

In [16]:
df_agua.orderBy(col("radiactividad_bq_l").desc()).show(5, truncate=False)


[Stage 30:===>                                                    (1 + 15) / 16]

+----------+---------+---------+-------------------+--------------------+----+------------+-------------+-------------------+------------------------------+---------------------+----------+-------------+-------------+-----------+----------------------------+--------------------------+-------------------+------------------+----------+-------------------------+
|id_lectura|sensor_id|ubicacion|fecha_hora         |canal_transmision_id|ph  |turbidez_ntu|temperatura_c|conductividad_us_cm|solidos_disueltos_totales_mg_l|oxigeno_disuelto_mg_l|plomo_mg_l|arsenico_mg_l|mercurio_mg_l|cadmio_mg_l|coliformes_fecales_nmp_100ml|escherichia_coli_nmp_100ml|presencia_parasitos|radiactividad_bq_l|caudal_l_s|indice_riesgo_normalizado|
+----------+---------+---------+-------------------+--------------------+----+------------+-------------+-------------------+------------------------------+---------------------+----------+-------------+-------------+-----------+----------------------------+------------------

In [17]:
df_agua.orderBy(col("ubicacion").asc(), col("plomo_mg_l").desc()).show(5, truncate=False)

[Stage 31:===>                                                    (1 + 15) / 16]

+----------+---------+-------------------+-------------------+--------------------+----+------------+-------------+-------------------+------------------------------+---------------------+----------+-------------+-------------+-----------+----------------------------+--------------------------+-------------------+------------------+----------+-------------------------+
|id_lectura|sensor_id|ubicacion          |fecha_hora         |canal_transmision_id|ph  |turbidez_ntu|temperatura_c|conductividad_us_cm|solidos_disueltos_totales_mg_l|oxigeno_disuelto_mg_l|plomo_mg_l|arsenico_mg_l|mercurio_mg_l|cadmio_mg_l|coliformes_fecales_nmp_100ml|escherichia_coli_nmp_100ml|presencia_parasitos|radiactividad_bq_l|caudal_l_s|indice_riesgo_normalizado|
+----------+---------+-------------------+-------------------+--------------------+----+------------+-------------+-------------------+------------------------------+---------------------+----------+-------------+-------------+-----------+-----------------

In [18]:
# sort() es alias de orderBy()
df_agua.sort(col("ph").asc_nulls_last()).show(5, truncate=False)

[Stage 32:>                                                       (0 + 16) / 16]

+----------+---------+---------+-------------------+--------------------+----+------------+-------------+-------------------+------------------------------+---------------------+----------+-------------+-------------+-----------+----------------------------+--------------------------+-------------------+------------------+----------+-------------------------+
|id_lectura|sensor_id|ubicacion|fecha_hora         |canal_transmision_id|ph  |turbidez_ntu|temperatura_c|conductividad_us_cm|solidos_disueltos_totales_mg_l|oxigeno_disuelto_mg_l|plomo_mg_l|arsenico_mg_l|mercurio_mg_l|cadmio_mg_l|coliformes_fecales_nmp_100ml|escherichia_coli_nmp_100ml|presencia_parasitos|radiactividad_bq_l|caudal_l_s|indice_riesgo_normalizado|
+----------+---------+---------+-------------------+--------------------+----+------------+-------------+-------------------+------------------------------+---------------------+----------+-------------+-------------+-----------+----------------------------+------------------

## 6. Tratamiento de duplicados [control de calidad #3]

Diagnóstico primero, sin eliminar nada — ¿hay `id_lectura` repetidos? ¿mismo sensor
con la misma marca de tiempo (retransmisión duplicada)?

In [19]:
df_agua.groupBy("id_lectura").count().filter("count > 1").show()
df_agua.groupBy("sensor_id", "fecha_hora").count().filter("count > 1").show()

+----------+-----+
|id_lectura|count|
+----------+-----+
+----------+-----+



[Stage 38:===>                                                    (1 + 15) / 16]

+---------+-------------------+-----+
|sensor_id|         fecha_hora|count|
+---------+-------------------+-----+
|     0156|2025-06-11 09:00:00|    3|
|     0119|2025-04-17 04:00:00|    2|
|     0230|2025-04-10 00:00:00|    2|
|     0185|2024-03-01 14:00:00|    2|
|     0195|2024-07-04 04:00:00|    2|
|     0094|2024-01-07 04:00:00|    2|
|     0193|2024-10-29 15:00:00|    2|
|     0114|2024-06-26 15:00:00|    2|
|     0068|2025-09-12 09:00:00|    2|
|     0140|2025-08-26 07:00:00|    2|
|     0111|2025-06-12 12:00:00|    2|
|     0237|2025-05-22 20:00:00|    3|
|     0092|2025-03-18 16:00:00|    2|
|     0133|2025-01-17 16:00:00|    2|
|     0065|2024-11-14 23:00:00|    2|
|     0108|2025-03-21 11:00:00|    2|
|     0059|2025-08-30 19:00:00|    2|
|     0043|2025-09-16 18:00:00|    2|
|     0081|2024-01-12 04:00:00|    2|
|     0211|2024-11-25 00:00:00|    2|
+---------+-------------------+-----+
only showing top 20 rows


**Técnica 1 — `dropDuplicates()` / `distinct()`:**

In [20]:
total = df_agua.count()
sin_dup_fila_completa = df_agua.distinct().count()
sin_dup_por_lectura = df_agua.dropDuplicates(["id_lectura"]).count()
sin_dup_sensor_fecha = df_agua.dropDuplicates(["sensor_id", "fecha_hora"]).count()

print(f"Total: {total}")
print(f"Sin duplicar (fila completa): {sin_dup_fila_completa}")
print(f"Sin duplicar (por id_lectura): {sin_dup_por_lectura}")
print(f"Sin duplicar (por sensor_id+fecha_hora): {sin_dup_sensor_fecha}")

[Stage 56:===>                                                    (1 + 15) / 16]

Total: 1500000
Sin duplicar (fila completa): 1500000
Sin duplicar (por id_lectura): 1500000
Sin duplicar (por sensor_id+fecha_hora): 1313822


**Técnica 2 — `Window + row_number()`:** si hubiera lecturas duplicadas por
`sensor_id`+`fecha_hora`, se conserva la de mayor `id_lectura` (la más reciente en el
orden de ingestión) — `id_lectura` es único por fila, no hace falta columna de desempate.

In [21]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("sensor_id", "fecha_hora").orderBy(
    col("id_lectura").desc()
)

df_ranked = df_agua.withColumn("row_num", row_number().over(window_spec))
df_sin_duplicados = df_ranked.filter(col("row_num") == 1).drop("row_num")

print(f"Filas originales: {df_agua.count()}, tras Window+row_number(): {df_sin_duplicados.count()}")


[Stage 65:===>                                                    (1 + 15) / 16]

Filas originales: 1500000, tras Window+row_number(): 1313822


## 7. Tratar nulos con `.na.fill()` y `.na.drop()` [cierra Silver]

Se rellena donde el nulo tiene un significado razonable; no se rellena con un valor
falso una columna donde eso inventaría un dato (por ejemplo, no se rellena `ph` con 0).

In [22]:
df_agua_limpio = df_agua.na.fill({
    "presencia_parasitos": 0,      # bandera: nulo = no detectado, razonable rellenar con 0
    "canal_transmision_id": -1,    # marca "canal desconocido", no se confunde con un canal real
})

`sensor_id`/`id_lectura` son las columnas críticas — sin ellas, la lectura no se puede vincular a ningún sensor. Solo esas filas se eliminan, no toda fila con cualquier nulo:

In [23]:
# Referencia: qué tan agresivo es na.drop() sin argumentos (no se usa como version final)
print("Filas si se usa na.drop() sin argumentos:", df_agua.na.drop().count())

df_agua_valido = df_agua_limpio.na.drop(subset=["sensor_id", "id_lectura"])

print(f"Filas antes: {df_agua.count()}, despues de na.drop(subset=[...]): {df_agua_valido.count()}")

assert df_agua_valido.filter(col("sensor_id").isNull()).count() == 0

Filas si se usa na.drop() sin argumentos: 1382430


Filas antes: 1500000, despues de na.drop(subset=[...]): 1500000


In [24]:
df_agua_valido = df_agua_valido.cache()

> Con esto, `df_agua_valido` es **Silver**: esquema validado + nulos tratados + duplicados resueltos.

## 8. Escritura en múltiples formatos

In [25]:
muestra = df_agua_valido.limit(100)

muestra.write.format("csv").option("header", True).mode("overwrite").save(f"{ARTIFACTS}/muestra_csv")
muestra.write.format("json").mode("overwrite").save(f"{ARTIFACTS}/muestra_json")
muestra.write.format("parquet").mode("overwrite").save(f"{ARTIFACTS}/muestra_parquet")

## 9. Escritura particionada en Parquet [Silver → Gold]

Columna de partición: `ubicacion` — solo 4 valores distintos, y es justo la columna
por la que se filtra en el análisis del proyecto (cardinalidad baja, uso frecuente en
filtros).

In [26]:
(
    df_agua_valido
    .repartition(4)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("ubicacion")
    .save(f"{ARTIFACTS}/lecturas_particionado")
)

In [27]:
import os
for carpeta in sorted(os.listdir(f"{ARTIFACTS}/lecturas_particionado")):
    print(carpeta)

._SUCCESS.crc
_SUCCESS
ubicacion=Cabanillas (ciudad)
ubicacion=Cabanillas (nacimiento de agua)
ubicacion=Juliaca (urbano)
ubicacion=Rio Coata


## 10. Leer de vuelta y verificar el particionamiento [Gold]

In [28]:
df_verificacion = spark.read.parquet(f"{ARTIFACTS}/lecturas_particionado")
df_verificacion.printSchema()
df_verificacion.count()

root
 |-- id_lectura: integer (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- fecha_hora: timestamp (nullable = true)
 |-- canal_transmision_id: integer (nullable = true)
 |-- ph: double (nullable = true)
 |-- turbidez_ntu: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- conductividad_us_cm: double (nullable = true)
 |-- solidos_disueltos_totales_mg_l: double (nullable = true)
 |-- oxigeno_disuelto_mg_l: double (nullable = true)
 |-- plomo_mg_l: double (nullable = true)
 |-- arsenico_mg_l: double (nullable = true)
 |-- mercurio_mg_l: double (nullable = true)
 |-- cadmio_mg_l: double (nullable = true)
 |-- coliformes_fecales_nmp_100ml: double (nullable = true)
 |-- escherichia_coli_nmp_100ml: double (nullable = true)
 |-- presencia_parasitos: integer (nullable = true)
 |-- radiactividad_bq_l: double (nullable = true)
 |-- caudal_l_s: double (nullable = true)
 |-- indice_riesgo_normalizado: double (nullable = true)
 |-- ubicacion: string (nullab

1500000

In [29]:
assert df_verificacion.count() == df_agua_valido.count()
print("Verificacion OK: el conteo coincide.")

Verificacion OK: el conteo coincide.


`explain(True)` filtrando por la columna particionada — debe aparecer `PartitionFilters`:

In [30]:
df_verificacion.filter(col("ubicacion") == "Rio Coata").explain(True)

== Parsed Logical Plan ==
'Filter '`=`('ubicacion, Rio Coata)
+- Relation [id_lectura#3545,sensor_id#3546,fecha_hora#3547,canal_transmision_id#3548,ph#3549,turbidez_ntu#3550,temperatura_c#3551,conductividad_us_cm#3552,solidos_disueltos_totales_mg_l#3553,oxigeno_disuelto_mg_l#3554,plomo_mg_l#3555,arsenico_mg_l#3556,mercurio_mg_l#3557,cadmio_mg_l#3558,coliformes_fecales_nmp_100ml#3559,escherichia_coli_nmp_100ml#3560,presencia_parasitos#3561,radiactividad_bq_l#3562,caudal_l_s#3563,indice_riesgo_normalizado#3564,ubicacion#3565] parquet

== Analyzed Logical Plan ==
id_lectura: int, sensor_id: string, fecha_hora: timestamp, canal_transmision_id: int, ph: double, turbidez_ntu: double, temperatura_c: double, conductividad_us_cm: double, solidos_disueltos_totales_mg_l: double, oxigeno_disuelto_mg_l: double, plomo_mg_l: double, arsenico_mg_l: double, mercurio_mg_l: double, cadmio_mg_l: double, coliformes_fecales_nmp_100ml: double, escherichia_coli_nmp_100ml: double, presencia_parasitos: int, rad

Balance de filas por partición:

In [31]:
df_verificacion.groupBy("ubicacion").count().orderBy("ubicacion").show(truncate=False)

+-------------------------------+------+
|ubicacion                      |count |
+-------------------------------+------+
|Cabanillas (ciudad)            |360166|
|Cabanillas (nacimiento de agua)|299748|
|Juliaca (urbano)               |420063|
|Rio Coata                      |420023|
+-------------------------------+------+



In [32]:
df_agua_valido.unpersist()

DataFrame[id_lectura: int, sensor_id: string, ubicacion: string, fecha_hora: timestamp, canal_transmision_id: int, ph: double, turbidez_ntu: double, temperatura_c: double, conductividad_us_cm: double, solidos_disueltos_totales_mg_l: double, oxigeno_disuelto_mg_l: double, plomo_mg_l: double, arsenico_mg_l: double, mercurio_mg_l: double, cadmio_mg_l: double, coliformes_fecales_nmp_100ml: double, escherichia_coli_nmp_100ml: double, presencia_parasitos: int, radiactividad_bq_l: double, caudal_l_s: double, indice_riesgo_normalizado: double]